<a href="https://colab.research.google.com/github/Adamphoenix003/GNN-LinkPrediction/blob/main/SFDDGNN4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           *args, "-q"])

pip("numpy==1.26.4", "--force-reinstall")
pip("torch-geometric")
pip("node2vec")
pip("scikit-learn")
pip("scipy")
pip("gensim")

print("✅ Done. Runtime → Restart Runtime, then Cell 2.")

✅ Done. Runtime → Restart Runtime, then Cell 2.


In [ ]:
import numpy as np
print(f"numpy : {np.__version__}")

import os, random, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import SAGEConv
from torch_geometric.utils import k_hop_subgraph
from torch_geometric.data import Data, Batch
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr

from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.sparse import coo_matrix
import networkx as nx

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch={torch.__version__}  pyg={torch_geometric.__version__}  "
      f"device={DEVICE}")
print("✅ Imports done")

numpy : 2.0.2
torch=2.10.0+cpu  pyg=2.7.0  device=cpu
✅ Imports done


In [ ]:
ROOT    = "./data"
dataset = Planetoid(root=ROOT, name="Cora")
data    = dataset[0]

torch.manual_seed(SEED)
transform = RandomLinkSplit(
    num_val                    = 0.10,
    num_test                   = 0.10,
    is_undirected              = True,
    add_negative_train_samples = True,
    neg_sampling_ratio         = 1.0,
)
train_data, val_data, test_data = transform(data)

num_nodes  = data.num_nodes
edge_index = train_data.edge_index          # (2, E_train)

print(f"Nodes={num_nodes}  Train edges={edge_index.shape[1]}")
print(f"Val  pairs={val_data.edge_label_index.shape[1]}")
print(f"Test pairs={test_data.edge_label_index.shape[1]}")
print("✅ Cora loaded")


In [ ]:
# ── ELSE: DeepWalk + Node2Vec + MF (eq. 1-2) ────────
#
#  Paper: "we use DeepWalk, Node2Vec and MF for link prediction
#          to train the basis methods of ELSE"
#  Input to all three: one-hot encoding X^O (identity matrix)
#  Output: X^B = MLP( ||_B B(A, X^O) )  dim=128

EMB_DIM = 64   # per basis method → total = 192 before MLP

# ── MF via SVD ────────────────────────────────────────────────
def train_mf(edge_index_cpu, num_nodes, dim=64):
    src, dst = edge_index_cpu.numpy()
    rows = np.concatenate([src, dst])
    cols = np.concatenate([dst, src])
    vals = np.ones(len(rows), dtype=np.float32)
    A    = coo_matrix((vals, (rows, cols)),
                      shape=(num_nodes, num_nodes))
    svd  = TruncatedSVD(n_components=dim, random_state=SEED)
    emb  = svd.fit_transform(A)
    return normalize(emb).astype(np.float32)

# ── DeepWalk (p=1, q=1) ───────────────────────────────────────
def train_deepwalk(edge_index_cpu, num_nodes, dim=64):
    from node2vec import Node2Vec
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    G.add_edges_from(edge_index_cpu.t().tolist())
    m = Node2Vec(G, dimensions=dim, walk_length=20, num_walks=10,
                 p=1, q=1, workers=1, seed=SEED, quiet=True)
    w = m.fit(window=5, min_count=1, batch_words=4)
    emb = np.zeros((num_nodes, dim), dtype=np.float32)
    for i in range(num_nodes):
        if str(i) in w.wv:
            emb[i] = w.wv[str(i)]
    return normalize(emb).astype(np.float32)

# ── Node2Vec (p=0.25, q=4) ────────────────────────────────────
def train_node2vec(edge_index_cpu, num_nodes, dim=64):
    from node2vec import Node2Vec
    G = nx.Graph()
    G.add_nodes_from(range(num_nodes))
    G.add_edges_from(edge_index_cpu.t().tolist())
    m = Node2Vec(G, dimensions=dim, walk_length=20, num_walks=10,
                 p=0.25, q=4, workers=1, seed=SEED, quiet=True)
    w = m.fit(window=5, min_count=1, batch_words=4)
    emb = np.zeros((num_nodes, dim), dtype=np.float32)
    for i in range(num_nodes):
        if str(i) in w.wv:
            emb[i] = w.wv[str(i)]
    return normalize(emb).astype(np.float32)

print("Training ELSE basis methods…")
print("  [1/3] MF…")
mf_emb  = train_mf(edge_index.cpu(), num_nodes, dim=EMB_DIM)
print("  [2/3] DeepWalk…")
dw_emb  = train_deepwalk(edge_index.cpu(), num_nodes, dim=EMB_DIM)
print("  [3/3] Node2Vec…")
n2v_emb = train_node2vec(edge_index.cpu(), num_nodes, dim=EMB_DIM)

# Concatenate: (N, 192)  then store as CPU tensor
basis_np = np.concatenate([mf_emb, dw_emb, n2v_emb], axis=1)
basis_t  = torch.from_numpy(basis_np)          # CPU, moved per-subgraph
print(f"ELSE basis: {basis_t.shape}")
print("✅ ELSE basis done")

Training ELSE basis methods…
  [1/3] MF…
  [2/3] DeepWalk…
  [3/3] Node2Vec…
ELSE basis: torch.Size([2708, 192])
✅ ELSE basis done


In [ ]:
class ELSE_MLP(nn.Module):
    def __init__(self, in_dim=192, hidden=256, out_dim=128, dropout=0.5):
        super().__init__()
        self.fc1  = nn.Linear(in_dim, hidden)
        self.drop = nn.Dropout(dropout)
        self.fc2  = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(self.drop(F.relu(self.fc1(x))))

else_mlp = ELSE_MLP(in_dim=192, hidden=256, out_dim=128, dropout=0.5
                    ).to(DEVICE)
print(f"ELSE MLP params={sum(p.numel() for p in else_mlp.parameters()):,}")
print("✅ ELSE MLP ready")


ELSE MLP params=82,304
✅ ELSE MLP ready


In [ ]:
# ── ── K-hop Subgraph Dataset ─────────────────────────
#
#  Paper Section 3.1.2 (KEY):
#  "we use the subgraph composed of node pairs and their
#   k-hop neighborhoods as the training data"
#
#  For each target edge (v_i, v_j):
#    - Extract all nodes within k hops of BOTH v_i and v_j
#    - Extract the induced subgraph on those nodes
#    - Re-index nodes locally (0..M-1)
#    - Record the local indices of v_i and v_j (the target pair)
#    - Store the ELSE embedding slice for those M nodes
#
#  K=2 (2-hop neighbourhood) following SEAL convention and
#  paper's reference to "k-hop neighbors N^k_i"

K_HOPS = 2   # paper uses k-hop; we use k=2 matching Section 3.1

class SubgraphDataset(Dataset):
    """
    For each target edge pair (v_i, v_j):
      1. Extract the union of k-hop neighbourhoods of v_i and v_j
      2. Build the induced subgraph
      3. Return local subgraph data + local indices of the pair
    """

    def __init__(self, edge_label_index, edge_label,
                 full_edge_index, basis_np, num_nodes,
                 k=2, max_nodes_per_subgraph=200):
        """
        edge_label_index : (2, P) target pairs
        edge_label       : (P,)   1=positive, 0=negative
        full_edge_index  : (2, E) full training graph
        basis_np         : (N, 192) ELSE basis embeddings (numpy)
        """
        self.pairs     = edge_label_index.t().tolist()   # list of [i,j]
        self.labels    = edge_label.tolist()
        self.ei        = full_edge_index                 # (2, E) cpu
        self.basis     = basis_np                        # (N, 192) numpy
        self.N         = num_nodes
        self.k         = k
        self.max_nodes = max_nodes_per_subgraph

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        src_node = self.pairs[idx][0]
        dst_node = self.pairs[idx][1]
        label    = self.labels[idx]

        # ── Extract k-hop subgraph around src AND dst ─────────
        # k_hop_subgraph returns:
        #   subset       : node indices in the subgraph
        #   sub_ei       : re-indexed edge_index within subgraph
        #   mapping      : positions of [src, dst] in subset
        #   edge_mask    : which original edges are in subgraph
        subset_src, sub_ei_src, mapping_src, _ = k_hop_subgraph(
            node_idx   = src_node,
            num_hops   = self.k,
            edge_index = self.ei,
            relabel_nodes = True,
            num_nodes  = self.N,
        )
        subset_dst, sub_ei_dst, mapping_dst, _ = k_hop_subgraph(
            node_idx   = dst_node,
            num_hops   = self.k,
            edge_index = self.ei,
            relabel_nodes = True,
            num_nodes  = self.N,
        )

        # ── Union of the two neighbourhoods ───────────────────
        # Combine node sets from both k-hop neighbourhoods
        union_nodes = torch.cat([subset_src, subset_dst]).unique()

        # Limit subgraph size to avoid OOM
        if union_nodes.shape[0] > self.max_nodes:
            # Keep src, dst, and random sample of the rest
            others = union_nodes[
                (union_nodes != src_node) & (union_nodes != dst_node)
            ]
            perm    = torch.randperm(others.shape[0])
            others  = others[perm[:self.max_nodes - 2]]
            union_nodes = torch.cat([
                torch.tensor([src_node, dst_node]),
                others
            ]).unique()

        # ── Build induced subgraph on union_nodes ─────────────
        # Re-index union_nodes to 0..M-1
        union_nodes_sorted, _ = union_nodes.sort()
        M = union_nodes_sorted.shape[0]

        # Build a mapping from original node id → local id
        # Using a lookup tensor for speed
        node2local = torch.full((self.N,), -1, dtype=torch.long)
        node2local[union_nodes_sorted] = torch.arange(M)

        # Filter edges: keep only edges where both endpoints in union
        ei = self.ei                                     # (2, E)
        mask = (node2local[ei[0]] >= 0) & (node2local[ei[1]] >= 0)
        sub_ei = node2local[ei[:, mask]]                 # (2, E_sub)

        # ── Local indices of the target pair ──────────────────
        local_src = node2local[src_node].item()
        local_dst = node2local[dst_node].item()

        # ── ELSE features for the subgraph nodes ──────────────
        sub_x = torch.from_numpy(
            self.basis[union_nodes_sorted.numpy()]       # (M, 192)
        ).float()

        return {
            "x":         sub_x,            # (M, 192)
            "edge_index": sub_ei,           # (2, E_sub)
            "src":       local_src,         # int
            "dst":       local_dst,         # int
            "label":     float(label),      # float
            "num_nodes": M,                 # int
        }


def collate_subgraphs(batch):
    """
    Manually batch subgraphs by offsetting node indices.
    Returns batched x, edge_index, src_list, dst_list, labels.
    """
    xs         = []
    eis        = []
    srcs       = []
    dsts       = []
    labels     = []
    node_offset = 0

    for item in batch:
        xs.append(item["x"])
        # Offset edge_index by current node count
        eis.append(item["edge_index"] + node_offset)
        srcs.append(item["src"]   + node_offset)
        dsts.append(item["dst"]   + node_offset)
        labels.append(item["label"])
        node_offset += item["num_nodes"]

    return {
        "x":          torch.cat(xs,  dim=0),           # (total_N, 192)
        "edge_index": torch.cat(eis, dim=1),            # (2, total_E)
        "srcs":       torch.tensor(srcs, dtype=torch.long),
        "dsts":       torch.tensor(dsts, dtype=torch.long),
        "labels":     torch.tensor(labels, dtype=torch.float),
    }


# Build datasets
print("Building subgraph datasets (k=2)…")

train_sg_ds = SubgraphDataset(
    edge_label_index       = train_data.edge_label_index.cpu(),
    edge_label             = train_data.edge_label.cpu(),
    full_edge_index        = edge_index.cpu(),
    basis_np               = basis_np,
    num_nodes              = num_nodes,
    k                      = K_HOPS,
    max_nodes_per_subgraph = 200,
)
val_sg_ds = SubgraphDataset(
    edge_label_index       = val_data.edge_label_index.cpu(),
    edge_label             = val_data.edge_label.cpu(),
    full_edge_index        = edge_index.cpu(),
    basis_np               = basis_np,
    num_nodes              = num_nodes,
    k                      = K_HOPS,
    max_nodes_per_subgraph = 200,
)
test_sg_ds = SubgraphDataset(
    edge_label_index       = test_data.edge_label_index.cpu(),
    edge_label             = test_data.edge_label.cpu(),
    full_edge_index        = edge_index.cpu(),
    basis_np               = basis_np,
    num_nodes              = num_nodes,
    k                      = K_HOPS,
    max_nodes_per_subgraph = 200,
)

# Use batch_size=32 for subgraph batching
# (each subgraph has ~50-200 nodes, so 32 subgraphs ≈ 1024-6400 nodes)
BATCH_SG = 32

train_sg_loader = DataLoader(
    train_sg_ds, batch_size=BATCH_SG, shuffle=True,
    num_workers=2, collate_fn=collate_subgraphs,
    pin_memory=True,
)
val_sg_loader = DataLoader(
    val_sg_ds, batch_size=BATCH_SG, shuffle=False,
    num_workers=2, collate_fn=collate_subgraphs,
)
test_sg_loader = DataLoader(
    test_sg_ds, batch_size=BATCH_SG, shuffle=False,
    num_workers=2, collate_fn=collate_subgraphs,
)

print(f"  Train batches : {len(train_sg_loader)}")
print(f"  Val   batches : {len(val_sg_loader)}")
print(f"  Test  batches : {len(test_sg_loader)}")
print("✅ Subgraph datasets ready")


Building subgraph datasets (k=2)…
  Train batches : 264
  Val   batches : 33
  Test  batches : 33
✅ Subgraph datasets ready


In [ ]:
# ── GraphSAGE Structure Encoder (eq. 3-5) ──────────
#
#  Runs ON the extracted subgraph.
#  Node update   (eq. 3): SAGEConv × 2
#  Edge update   (eq. 4): W_E · Aggregator(h^{k-1}_ij, h^k_i, h^k_j)
#  ReadOut       (eq. 5): p_ij = δ(h_ij)

class GraphSAGE_StructureEncoder(nn.Module):
    """
    Operates on a local k-hop subgraph.
    Input x: ELSE embeddings of subgraph nodes (M, 128)
             (after ELSE_MLP projects 192→128)
    """
    def __init__(self, in_dim=128, hidden=256,
                 out_dim=128, dropout=0.5):
        super().__init__()
        # W_N: node update (eq. 3)
        self.sage1 = SAGEConv(in_dim,  hidden)
        self.sage2 = SAGEConv(hidden,  out_dim)

        # W_E: edge update (eq. 4)
        self.edge_up1 = nn.Linear(in_dim*2  + hidden*2,  hidden)
        self.edge_up2 = nn.Linear(hidden    + out_dim*2, out_dim)

        # ReadOut δ(h_ij) → p_ij  (eq. 5)
        self.readout = nn.Sequential(
            nn.Linear(out_dim, out_dim // 2),
            nn.ReLU(),
            nn.Linear(out_dim // 2, 1),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, edge_index, src_idx, dst_idx):
        """
        x          : (M, in_dim)  node features on the subgraph
        edge_index : (2, E_sub)   subgraph edges (re-indexed)
        src_idx    : (B,)         local indices of source nodes
        dst_idx    : (B,)         local indices of dest nodes

        Returns:
          logits : (B,)  ReadOut scores
          h2     : (M, out_dim)  node embeddings
        """
        h0 = x   # (M, in_dim)

        # ── Layer 1 node update ───────────────────────────────
        h1 = F.relu(self.sage1(h0, edge_index))   # (M, hidden)
        h1 = self.drop(h1)

        # ── Layer 1 edge update ───────────────────────────────
        # h^0_ij = x^S_i || x^S_j
        e0    = torch.cat([h0[src_idx], h0[dst_idx]], dim=-1)  # (B, 2*in)
        e1_in = torch.cat([e0, h1[src_idx], h1[dst_idx]], -1)  # (B, 2in+2hid)
        e1    = F.relu(self.edge_up1(e1_in))                   # (B, hidden)

        # ── Layer 2 node update ───────────────────────────────
        h2 = F.relu(self.sage2(h1, edge_index))   # (M, out_dim)
        h2 = self.drop(h2)

        # ── Layer 2 edge update ───────────────────────────────
        e2_in = torch.cat([e1, h2[src_idx], h2[dst_idx]], -1)  # (B, hid+2out)
        e2    = F.relu(self.edge_up2(e2_in))                   # (B, out_dim)

        # ── ReadOut p_ij = δ(h_ij) ────────────────────────────
        logits = self.readout(e2).squeeze(-1)                  # (B,)

        return logits, h2


struct_enc = GraphSAGE_StructureEncoder(
    in_dim  = 128,
    hidden  = 256,
    out_dim = 128,
    dropout = 0.5,
).to(DEVICE)

total_params = (sum(p.numel() for p in else_mlp.parameters()) +
                sum(p.numel() for p in struct_enc.parameters()))
print(f"ELSE MLP + StructureEncoder params = {total_params:,}")
print("✅ GraphSAGE Structure Encoder ready")




ELSE MLP + StructureEncoder params = 484,609
✅ GraphSAGE Structure Encoder ready


In [ ]:
#  ── Training (paper hyperparameters) ────────────────
#
#  Adam lr=1e-3, L2=1e-5, dropout=0.5
#  Early stop patience=50
#  Loss eq. 5: BCE on subgraph edge scores

LR       = 1e-3
L2_REG   = 1e-5
PATIENCE = 50
MAX_EP   = 500

optimizer = Adam(
    list(else_mlp.parameters()) +
    list(struct_enc.parameters()),
    lr=LR, weight_decay=L2_REG,
)


def run_one_epoch(loader, train=True):
    """One pass through all subgraph batches."""
    if train:
        else_mlp.train(); struct_enc.train()
    else:
        else_mlp.eval();  struct_enc.eval()

    total_loss = 0.0
    all_probs  = []
    all_labels = []

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            x          = batch["x"].to(DEVICE)
            sub_ei     = batch["edge_index"].to(DEVICE)
            srcs       = batch["srcs"].to(DEVICE)
            dsts       = batch["dsts"].to(DEVICE)
            labels     = batch["labels"].to(DEVICE)

            # ELSE MLP: 192 → 128
            x_b = else_mlp(x)                        # (total_M, 128)

            # GraphSAGE on subgraph
            logits, _ = struct_enc(x_b, sub_ei,
                                   srcs, dsts)        # (B,)

            # Loss eq. 5
            loss = F.binary_cross_entropy_with_logits(logits, labels)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(else_mlp.parameters()) +
                    list(struct_enc.parameters()),
                    max_norm=1.0,
                )
                optimizer.step()

            total_loss += loss.item()
            all_probs.append(torch.sigmoid(logits).detach().cpu())
            all_labels.append(labels.detach().cpu())

    avg_loss   = total_loss / max(len(loader), 1)
    all_probs  = torch.cat(all_probs).numpy()
    all_labels = torch.cat(all_labels).numpy()
    ap  = average_precision_score(all_labels, all_probs)
    auc = roc_auc_score(all_labels, all_probs)
    return avg_loss, ap, auc


print(f"\nTraining (subgraph-based GraphSAGE)  "
      f"max={MAX_EP} epochs  patience={PATIENCE}\n")

best_val_ap  = 0.0
patience_cnt = 0
best_ckpt    = None

for epoch in range(1, MAX_EP + 1):
    train_loss, train_ap, train_auc = run_one_epoch(
        train_sg_loader, train=True
    )

    # Validate every 5 epochs
    if epoch % 5 == 0:
        val_loss, val_ap, val_auc = run_one_epoch(
            val_sg_loader, train=False
        )
        print(f"  Epoch {epoch:4d} | "
              f"train_loss={train_loss:.4f} "
              f"train_AP={train_ap*100:.2f}% | "
              f"val_AP={val_ap*100:.2f}%  "
              f"val_AUC={val_auc*100:.2f}%")

        if val_ap > best_val_ap:
            best_val_ap  = val_ap
            patience_cnt = 0
            best_ckpt = {
                "else_mlp":  {k: v.clone() for k, v in
                              else_mlp.state_dict().items()},
                "struct_enc": {k: v.clone() for k, v in
                               struct_enc.state_dict().items()},
            }
        else:
            patience_cnt += 5
            if patience_cnt >= PATIENCE:
                print(f"\n  Early stop  "
                      f"(best val AP={best_val_ap*100:.2f}%)")
                break

if best_ckpt:
    else_mlp.load_state_dict(best_ckpt["else_mlp"])
    struct_enc.load_state_dict(best_ckpt["struct_enc"])

print("\n✅ Training complete")





Training (subgraph-based GraphSAGE)  max=500 epochs  patience=50

  Epoch    5 | train_loss=0.2138 train_AP=95.25% | val_AP=77.86%  val_AUC=76.85%
  Epoch   10 | train_loss=0.1434 train_AP=97.84% | val_AP=79.60%  val_AUC=77.46%
  Epoch   15 | train_loss=0.1232 train_AP=98.27% | val_AP=79.70%  val_AUC=77.86%
  Epoch   20 | train_loss=0.1018 train_AP=98.97% | val_AP=79.68%  val_AUC=78.76%
  Epoch   25 | train_loss=0.0951 train_AP=99.13% | val_AP=79.35%  val_AUC=78.81%
  Epoch   30 | train_loss=0.0725 train_AP=99.54% | val_AP=79.59%  val_AUC=79.03%
  Epoch   35 | train_loss=0.0695 train_AP=99.59% | val_AP=81.80%  val_AUC=79.98%
  Epoch   40 | train_loss=0.0576 train_AP=99.60% | val_AP=81.43%  val_AUC=80.13%
  Epoch   45 | train_loss=0.0519 train_AP=99.77% | val_AP=82.41%  val_AUC=80.41%
  Epoch   50 | train_loss=0.0390 train_AP=99.89% | val_AP=80.60%  val_AUC=79.86%
  Epoch   55 | train_loss=0.0524 train_AP=99.74% | val_AP=82.64%  val_AUC=80.50%
  Epoch   60 | train_loss=0.0470 train_AP=

In [ ]:
#  ── Test evaluation ─────────────────────────────────

_, test_ap, test_auc = run_one_epoch(test_sg_loader, train=False)

print(f"\n{'='*52}")
print(f"  Pipeline 1 – Subgraph GraphSAGE – Test")
print(f"{'='*52}")
print(f"  AP  : {test_ap  * 100:.2f}%")
print(f"  AUC : {test_auc * 100:.2f}%")
print(f"{'='*52}")
print(f"  Paper target: AP=95.44%  AUC=94.73%")





  Pipeline 1 – Subgraph GraphSAGE – Test
  AP  : 84.12%
  AUC : 82.08%
  Paper target: AP=95.44%  AUC=94.73%


In [ ]:
# ─── Extract H_S for all nodes ─────────────────────
#
#  H_S = full-graph node embeddings used in the fusion gate.
#  We run a single full-graph forward pass (no subgraph) to get
#  one embedding per node — this is used by Pipeline 4.

else_mlp.eval(); struct_enc.eval()
with torch.no_grad():
    # Full-graph ELSE embedding
    X_B_full = else_mlp(basis_t.to(DEVICE))           # (N, 128)
    # Full-graph GraphSAGE node embeddings (no subgraph needed here)
    h1  = F.relu(struct_enc.sage1(X_B_full, edge_index.to(DEVICE)))
    H_S = F.relu(struct_enc.sage2(h1,       edge_index.to(DEVICE)))  # (N,128)

print(f"H_S shape : {H_S.shape}")
print(f"H_S NaN   : {torch.isnan(H_S).any().item()}")
print(f"H_S Inf   : {torch.isinf(H_S).any().item()}")

# Sanity: connected > unconnected cosine sim
pos_mask  = test_data.edge_label.bool()
tp        = test_data.edge_label_index
sim_pos   = (H_S[tp[0,  pos_mask]] * H_S[tp[1,  pos_mask]]).sum(-1).mean()
sim_neg   = (H_S[tp[0, ~pos_mask]] * H_S[tp[1, ~pos_mask]]).sum(-1).mean()
print(f"\n  Cosine sim pos={sim_pos:.4f}  neg={sim_neg:.4f}  "
      f"Δ={sim_pos-sim_neg:.4f}")




H_S shape : torch.Size([2708, 128])
H_S NaN   : False
H_S Inf   : False

  Cosine sim pos=10531.3818  neg=3210.0901  Δ=7321.2920


In [ ]:
#  ── Save ───────────────────────────────────────────

torch.save({
    "H_S":        H_S.cpu(),
    "basis_t":    basis_t,
    "basis_np":   basis_np,
    "else_mlp":   else_mlp.state_dict(),
    "struct_enc": struct_enc.state_dict(),
    "train_data": train_data,
    "val_data":   val_data,
    "test_data":  test_data,
    "num_nodes":  num_nodes,
    "edge_index": edge_index.cpu(),
    "test_ap":    test_ap,
    "test_auc":   test_auc,
}, "pipeline1_outputs.pt")

print("✅ Saved → pipeline1_outputs.pt")
print(f"\n  Test AP={test_ap*100:.2f}%  AUC={test_auc*100:.2f}%")
print("  Next → Pipeline 2 (Feature-Based)")


✅ Saved → pipeline1_outputs.pt

  Test AP=84.12%  AUC=82.08%
  Next → Pipeline 2 (Feature-Based)


PIPELINE 2

In [ ]:
import numpy as np
import os, random, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import average_precision_score, roc_auc_score
import networkx as nx
from collections import deque
from tqdm import tqdm

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={DEVICE}")
print("✅ Imports done")






device=cpu
✅ Imports done


In [ ]:
# ── Load Pipeline 1 outputs ─────────────────────────
#
#  Everything we need is in pipeline1_outputs.pt:
#    train_data, val_data, test_data  — PyG Data splits
#    edge_index                       — training graph topology
#    num_nodes                        — N = 2708 for Cora

p1 = torch.load(
    "pipeline1_outputs.pt",
    map_location="cpu",
    weights_only=False   # 🔥 important
)

train_data  = p1["train_data"]
val_data    = p1["val_data"]
test_data   = p1["test_data"]
num_nodes   = p1["num_nodes"]
edge_index  = p1["edge_index"]          # (2, E_train) cpu

print(f"Nodes={num_nodes}  Train edges={edge_index.shape[1]}")
print(f"Val  pairs={val_data.edge_label_index.shape[1]}")
print(f"Test pairs={test_data.edge_label_index.shape[1]}")
print("✅ Pipeline 1 outputs loaded")

In [ ]:
# ── Node features for Cora ──────────────────────────
#
#  Paper uses the raw text version of Cora (titles + abstracts).
#  Standard PyG Cora only has 1433-dim BoW features.
#
#  Strategy:
#    1. Try to load raw text from the He et al. processed version
#       (tape-arxiv23 style). If found, use as strings.
#    2. Otherwise fall back to the 1433-dim BoW vectors directly.
#       The FeatureEncoder MLP will project these the same way
#       RoBERTa's projection head would project [CLS] tokens.
#
#  Either way, node_feats ends up as a (N, FEAT_DIM) float tensor.

# ── CELL 3 ── Load Cora TEXT features ─────────────────────────

import pandas as pd

# Upload cora.content manually to Colab before running
# Format: paper_id word1 word2 ... word1433 class

content = pd.read_csv("cora.content", sep="\t", header=None)

# Extract text (convert BoW → pseudo text)
texts = []

for i in range(len(content)):
    words = content.iloc[i, 1:-1]
    text = " ".join([f"word{j}" for j, val in enumerate(words) if val == 1])
    texts.append(text)

print(f"Loaded {len(texts)} node texts")
print("Sample:", texts[0][:200])
print("✅ Text data ready")



Loaded 2708 node texts
Sample: word118 word125 word176 word252 word351 word456 word507 word521 word619 word648 word698 word702 word734 word845 word902 word1205 word1209 word1236 word1352 word1426
✅ Text data ready


In [ ]:
#  ── Build Triplet Dataset (eq. 7 / Section 3.2) ─────
#
#  For each edge (v_i, v_j) in the training graph:
#    positive pair = (v_i, v_j)
#    negative pair = (v_i, v_k) where v_k is unreachable or ≥3 hops
#
#  Negative sampling (paper Section 3.2):
#    1. Rank nodes by degree centrality → candidate pool
#    2. Alternating random + BFS-guided sampling
#    3. Cap at 5 negative pairs per central node
#    4. Exclude nodes lacking 3+ hop reachable neighbours

def bfs_distances(adj: dict, source: int, max_hops: int = 3) -> dict:
    """BFS up to max_hops from source. Returns {node: distance}."""
    dist  = {source: 0}
    queue = deque([source])
    while queue:
        u = queue.popleft()
        if dist[u] >= max_hops:
            continue
        for v in adj.get(u, []):
            if v not in dist:
                dist[v] = dist[u] + 1
                queue.append(v)
    return dist


def build_triplets(edge_index_cpu: torch.Tensor,
                   num_nodes: int,
                   max_neg_per_node: int = 5,
                   min_hop: int = 3) -> list:
    """
    Returns list of (anchor, positive, negative) integer triples.
    Mirrors the triplet construction in Section 3.2.
    """
    # Build adjacency list
    src_arr = edge_index_cpu[0].numpy()
    dst_arr = edge_index_cpu[1].numpy()
    adj = {i: [] for i in range(num_nodes)}
    for s, d in zip(src_arr, dst_arr):
        adj[s].append(d)
        adj[d].append(s)

    nodes = list(range(num_nodes))

    # Step 1: degree centrality ranking
    degrees       = {n: len(adj[n]) for n in nodes}
    ranked_nodes  = sorted(nodes, key=lambda n: degrees[n], reverse=True)

    # Pre-build negative pools per node
    neg_pool = {}
    for node in ranked_nodes:
        near = set(bfs_distances(adj, node, max_hops=min_hop - 1).keys())
        candidates = [n for n in nodes if n not in near]
        if not candidates:
            continue

        # BFS-guided candidates (≥ min_hop hops away)
        far_map = bfs_distances(adj, node, max_hops=6)
        bfs_far = [n for n, d in far_map.items() if d >= min_hop]

        random_cands = candidates.copy()
        random.shuffle(random_cands)

        sampled, ri, bi, toggle = [], 0, 0, True
        while len(sampled) < max_neg_per_node:
            if toggle and ri < len(random_cands):
                sampled.append(random_cands[ri]); ri += 1
            elif not toggle and bi < len(bfs_far):
                sampled.append(bfs_far[bi]);      bi += 1
            else:
                break
            toggle = not toggle

        if sampled:
            neg_pool[node] = sampled

    # Build triplets from training edges
    edges = list(zip(src_arr.tolist(), dst_arr.tolist()))
    triplets = []
    for (vi, vj) in edges:
        if vi in neg_pool:
            vk = random.choice(neg_pool[vi])
            triplets.append((vi, vj, vk))
        if vj in neg_pool:
            vk = random.choice(neg_pool[vj])
            triplets.append((vj, vi, vk))

    print(f"  Triplets built: {len(triplets):,}  "
          f"(from {len(edges):,} directed edges)")
    return triplets


print("Building triplet dataset…")
triplets = build_triplets(edge_index.cpu(), num_nodes)
print("✅ Triplets ready")

In [ ]:
# ── CELL 5 ── Triplet Dataset (TEXT version) ─────────────────

class TripletDataset(Dataset):
    def __init__(self, triplets, texts):
        self.triplets = triplets
        self.texts = texts

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        vi, vj, vk = self.triplets[idx]
        return (
            self.texts[vi],
            self.texts[vj],
            self.texts[vk],
        )

In [ ]:
# ── RoBERTa Feature Encoder ───────────────────────

from transformers import RobertaTokenizer, RobertaModel

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")
roberta   = RobertaModel.from_pretrained("roberta-base").to(DEVICE)

class FeatureEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(768, 128)

    def forward(self, texts):
        tokens = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(DEVICE)

        out = roberta(**tokens)
        cls = out.last_hidden_state[:, 0, :]   # CLS token

        emb = self.proj(cls)
        return F.normalize(emb, p=2, dim=-1)

feat_encoder = FeatureEncoder().to(DEVICE)

print("✅ RoBERTa Feature Encoder ready")

In [ ]:
# ── CELL 7 ── Contrastive Loss (Paper Eq. 8) ────────────────

class ContrastiveLoss(nn.Module):
    def forward(self, h_i, h_j, h_k):
        sim_ij = F.cosine_similarity(h_i, h_j)
        sim_ik = F.cosine_similarity(h_i, h_k)

        numerator   = torch.exp(sim_ij)
        denominator = torch.exp(sim_ij) + torch.exp(sim_ik)

        loss = -torch.log(numerator / denominator)
        return loss.mean()

criterion = ContrastiveLoss()

print("✅ Paper contrastive loss ready")

✅ Paper contrastive loss ready


In [ ]:
#  ── Training (paper hyperparameters) ────────────────
#
#  Adam lr=1e-3, decay 0.1 every 3 epochs (paper Section 4.4)
#  L2 = 1e-5, dropout = 0.5
#  Early stop patience = 50

LR       = 1e-3
L2_REG   = 1e-5
PATIENCE = 50
MAX_EP   = 500

optimizer = Adam(
    feat_encoder.parameters(),
    lr=LR, weight_decay=L2_REG,
)
scheduler = StepLR(optimizer, step_size=20, gamma=0.5)


def run_one_epoch_feat(loader, train=True):
    """One pass through triplet batches — mirrors P1 run_one_epoch."""
    feat_encoder.train() if train else feat_encoder.eval()

    total_loss = 0.0
    all_sims   = []    # cosine sim for positive pairs (proxy metric)
    all_ones   = []    # all labels = 1 for triplets

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x_i, x_j, x_k in loader:
            x_i = x_i.to(DEVICE)
            x_j = x_j.to(DEVICE)
            x_k = x_k.to(DEVICE)

            h_i = feat_encoder([str(t) for t in x_i])
            h_j = feat_encoder([str(t) for t in x_j])
            h_k = feat_encoder([str(t) for t in x_k])

            loss = criterion(h_i, h_j, h_k)

            if train:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    feat_encoder.parameters(), max_norm=1.0
                )
                optimizer.step()

            total_loss += loss.item()
            # Positive pair similarity as a proxy for embedding quality
            sim_pos = (h_i * h_j).sum(-1).detach().cpu()
            all_sims.append(sim_pos)
            all_ones.extend([1.0] * len(sim_pos))

    avg_loss = total_loss / max(len(loader), 1)
    avg_sim  = torch.cat(all_sims).mean().item()
    return avg_loss, avg_sim


print(f"\nTraining Feature Encoder  "
      f"max={MAX_EP} epochs  patience={PATIENCE}\n")

best_val_loss = float("inf")
patience_cnt  = 0
best_ckpt     = None

for epoch in range(1, MAX_EP + 1):
    train_loss, train_sim = run_one_epoch_feat(
        train_trip_loader, train=True
    )
    scheduler.step()

    # Validate every 5 epochs (mirrors Pipeline 1 cadence)
    if epoch % 5 == 0:
        val_loss, val_sim = run_one_epoch_feat(
            val_trip_loader, train=False
        )
        print(f"  Epoch {epoch:4d} | "
              f"train_loss={train_loss:.4f}  "
              f"train_pos_sim={train_sim:.4f} | "
              f"val_loss={val_loss:.4f}  "
              f"val_pos_sim={val_sim:.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            patience_cnt  = 0
            best_ckpt = {k: v.clone()
                         for k, v in feat_encoder.state_dict().items()}
        else:
            patience_cnt += 5
            if patience_cnt >= PATIENCE:
                print(f"\n  Early stop  "
                      f"(best val loss={best_val_loss:.4f})")
                break

if best_ckpt:
    feat_encoder.load_state_dict(best_ckpt)

print("\n✅ Feature Encoder training complete")




In [ ]:
feat_encoder.eval()

H_F_list = []

with torch.no_grad():
    for i in range(0, len(texts), 32):
        batch = texts[i:i+32]
        emb = feat_encoder(batch)
        H_F_list.append(emb.cpu())

H_F_all = torch.cat(H_F_list, dim=0)

In [ ]:
#  ── Test evaluation ─────────────────────────────────
#
#  Score test edge pairs using dot product of H_F embeddings.


feat_encoder.eval()
with torch.no_grad():
    # Embed all nodes at once (N=2708 is small enough)
    H_F_all = feat_encoder(node_feats.to(DEVICE))    # (N, 128)

    src = test_data.edge_label_index[0]
    dst = test_data.edge_label_index[1]

    # Cosine sim = dot product (L2-normed)
    feat_scores = (H_F_all[src] * H_F_all[dst]).sum(-1).cpu().numpy()
    feat_labels = test_data.edge_label.numpy()

feat_ap  = average_precision_score(feat_labels, feat_scores)
feat_auc = roc_auc_score(feat_labels, feat_scores)

print(f"\n{'='*52}")
print(f"  Pipeline 2 – Feature Encoder – Test")
print(f"{'='*52}")
print(f"  AP  : {feat_ap  * 100:.2f}%")
print(f"  AUC : {feat_auc * 100:.2f}%")
print(f"{'='*52}")
print(f"  Paper note: full SFDDGNN AP=95.44% (after fusion)")





  Pipeline 2 – Feature Encoder – Test
  AP  : 64.18%
  AUC : 68.76%
  Paper note: full SFDDGNN AP=95.44% (after fusion)


In [ ]:
# ──── Extract H_F for all nodes ─────────────────────
#
#  H_F = full-graph feature-only node embeddings used in the
#  fusion gate. One 128-dim embedding per node.


feat_encoder.eval()
with torch.no_grad():
    H_F = feat_encoder(node_feats.to(DEVICE))   # (N, 128) L2-normed

print(f"H_F shape : {H_F.shape}")
print(f"H_F NaN   : {torch.isnan(H_F).any().item()}")
print(f"H_F Inf   : {torch.isinf(H_F).any().item()}")

# Sanity: connected nodes should have higher feature similarity
pos_mask = test_data.edge_label.bool()
tp       = test_data.edge_label_index

sim_pos = (H_F[tp[0,  pos_mask]] * H_F[tp[1,  pos_mask]]).sum(-1).mean()
sim_neg = (H_F[tp[0, ~pos_mask]] * H_F[tp[1, ~pos_mask]]).sum(-1).mean()
print(f"\n  Cosine sim pos={sim_pos:.4f}  neg={sim_neg:.4f}  "
      f"Δ={sim_pos - sim_neg:.4f}")




H_F shape : torch.Size([2708, 128])
H_F NaN   : False
H_F Inf   : False

  Cosine sim pos=0.3090  neg=-0.0105  Δ=0.3195


In [ ]:
#  ── Save ───────────────────────────────────────────
#
#  Bundle H_F + all Pipeline 1 carry-forwards into one file
#  so Pipeline 3 (GraphGPT) and Pipeline 4 (Fusion Gate)
#  only need to load a single checkpoint.

torch.save({
    # ── Feature pipeline outputs ──────────────────────────────
    "H_F":          H_F.cpu(),
    "feat_encoder": feat_encoder.state_dict(),
    "feat_ap":      feat_ap,
    "feat_auc":     feat_auc,
    # ── Carry-forward from Pipeline 1 ────────────────────────
    "H_S":          p1["H_S"],
    "basis_t":      p1["basis_t"],
    "basis_np":     p1["basis_np"],
    "else_mlp":     p1["else_mlp"],
    "struct_enc":   p1["struct_enc"],
    "train_data":   p1["train_data"],
    "val_data":     p1["val_data"],
    "test_data":    p1["test_data"],
    "num_nodes":    p1["num_nodes"],
    "edge_index":   p1["edge_index"],
    "struct_ap":    p1["test_ap"],
    "struct_auc":   p1["test_auc"],
}, "pipeline2_outputs.pt")

print("✅ Saved → pipeline2_outputs.pt")
print(f"\n  Feature  AP={feat_ap*100:.2f}%  AUC={feat_auc*100:.2f}%")
print(f"  Struct   AP={p1['test_ap']*100:.2f}%  AUC={p1['test_auc']*100:.2f}%")
print("  Next → Pipeline 3 (Graph Data Distribution Analysis / GraphGPT)")


✅ Saved → pipeline2_outputs.pt

  Feature  AP=64.18%  AUC=68.76%
  Struct   AP=84.12%  AUC=82.08%
  Next → Pipeline 3 (Graph Data Distribution Analysis / GraphGPT)


PIPELINE 3

In [ ]:
# NOTE ON GraphGPT:
#  The paper uses GraphGPT (Tang et al., SIGIR 2024) as the
#  graph large model. GraphGPT requires a 7B-parameter LLaMA
#  backbone which cannot run on a free Colab GPU (needs ~14 GB
#  VRAM). We implement a faithful approximation using:
#    - GIN  (Graph Isomorphism Network) for structural patterns
#    - Laplacian positional encodings  for global graph position
#    - Learned graph-level pooling      for the full-graph summary
#  This matches what GraphGPT extracts functionally: a global
#  embedding that captures graph distribution and domain knowledge.
#
#
# ============================================================


#  ── Install dependencies ───────────────────────────
# Run, then Runtime → Restart Runtime, then Cell 2 onwards.

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           *args, "-q"])

pip("numpy==1.26.4", "--force-reinstall")
pip("torch-geometric")
pip("scikit-learn", "scipy")

print("✅ Pipeline 3 dependencies installed.")
print("👉 Runtime → Restart Runtime, then run Cell 2 onwards.")


✅ Pipeline 3 dependencies installed.
👉 Runtime → Restart Runtime, then run Cell 2 onwards.


In [ ]:
import numpy as np
print(f"numpy  : {np.__version__}")

import os, random, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam

import torch_geometric
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit
from torch_geometric.nn import GINConv, global_mean_pool, global_add_pool
from torch_geometric.utils import (
    get_laplacian, to_dense_adj, to_undirected,
    degree, add_self_loops
)
from torch_geometric.data import Data, Batch

from sklearn.metrics import average_precision_score, roc_auc_score
from scipy.sparse.linalg import eigsh
from scipy.sparse import csr_matrix

import networkx as nx

# ── Reproducibility ───────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch  : {torch.__version__}")
print(f"pyg    : {torch_geometric.__version__}")
print(f"device : {DEVICE}")
print("✅ Imports done")




numpy  : 1.26.4
torch  : 2.10.0+cu128
pyg    : 2.7.0
device : cuda
✅ Imports done


In [ ]:
from torch_geometric.datasets import Planetoid
from torch_geometric.transforms import RandomLinkSplit

dataset = Planetoid(root="/tmp/Cora", name="Cora")

data = dataset[0]

split = RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    add_negative_train_samples=False,
)

train_data, val_data, test_data = split(data)

edge_index = train_data.edge_index
num_nodes  = data.num_nodes

print("✅ Data loaded")

In [ ]:
# ─── Laplacian Positional Encodings (LPE) ───────────
#
#  GraphGPT uses positional embeddings derived from the graph's
#  spectral structure. We implement this as Laplacian eigenvectors
#  (the standard approach in GraphGPS, Graphormer, etc.).
#
#  For each node, the k smallest non-trivial eigenvectors of the
#  normalised Laplacian give a position vector that encodes:
#    - Local neighbourhood density
#    - Global graph community structure
#    - Distance relationships between nodes
#
#  This is the positional embedding block shown in Fig 1 of the paper
#  (the "Position Embedding" box between GraphGPT and concatenation).

def compute_laplacian_pe(edge_index, num_nodes, k=20):
    """
    Compute k-dim Laplacian positional encodings for all nodes.
    Returns (num_nodes, k) tensor.

    k = number of eigenvectors (paper uses 128-d graph embedding,
    we use k=20 here and project to 128 in the GIN encoder).
    """
    # Build normalised Laplacian L = I - D^{-1/2} A D^{-1/2}
    edge_index_sym = to_undirected(edge_index, num_nodes=num_nodes)

    # Compute Laplacian edge_index and edge_weight
    lap_idx, lap_weight = get_laplacian(
        edge_index_sym,
        normalization = "sym",       # normalised Laplacian
        num_nodes     = num_nodes,
    )

    # Build sparse matrix for eigsh
    row = lap_idx[0].numpy()
    col = lap_idx[1].numpy()
    val = lap_weight.numpy()
    L   = csr_matrix((val, (row, col)), shape=(num_nodes, num_nodes))

    # k+1 smallest eigenvalues/vectors (skip the trivial 0 eigenvalue)
    k_actual = min(k + 1, num_nodes - 1)
    try:
        eigenvalues, eigenvectors = eigsh(L, k=k_actual, which="SM",
                                          tol=1e-3, maxiter=1000)
    except Exception:
        # Fallback: random positional encoding if eigsh fails
        print("  ⚠ eigsh failed, using random PE as fallback")
        return torch.randn(num_nodes, k) * 0.1

    # Drop the first (trivial) eigenvector, keep k
    pe = eigenvectors[:, 1 : k + 1]                   # (N, k)

    # Sign ambiguity fix: make each eigenvector's max-abs element positive
    signs = np.sign(pe[np.abs(pe).argmax(axis=0), np.arange(pe.shape[1])])
    pe    = pe * signs[np.newaxis, :]

    # Pad if we got fewer than k eigenvectors
    if pe.shape[1] < k:
        pad = np.zeros((num_nodes, k - pe.shape[1]), dtype=pe.dtype)
        pe  = np.concatenate([pe, pad], axis=1)

    return torch.from_numpy(pe.astype(np.float32))     # (N, k)


print("Computing Laplacian positional encodings (k=20)…")
PE_DIM = 20
lpe    = compute_laplacian_pe(edge_index, num_nodes, k=PE_DIM)
print(f"✅ LPE shape : {lpe.shape}  |  "
      f"min={lpe.min():.4f}  max={lpe.max():.4f}")




Computing Laplacian positional encodings (k=20)…
✅ LPE shape : torch.Size([2708, 20])  |  min=-0.2991  max=0.5098


In [ ]:
from torch_geometric.utils import degree

row, _ = edge_index
deg = degree(row, num_nodes=num_nodes).unsqueeze(-1)

deg = deg / (deg.max() + 1e-8)


rand_feat = torch.randn(num_nodes, 64)

node_feats = torch.cat([
    rand_feat,   # 64
    lpe,         # 20
    deg          # 1
], dim=-1)       # total = 85

NODE_FEAT_DIM = node_feats.shape[1]

print("Node feature dim:", NODE_FEAT_DIM)

Node feature dim: 85


In [ ]:

#  ── GIN-based Graph Encoder (GraphGPT approximation) ─
#
#  Graph Isomorphism Network-  We use a 3-layer GIN with:
#    - Batch normalisation after each layer
#    - Residual connections
#    - Hierarchical readout (mean + add pooling concatenated)
#    - A final projection to 128-d  → this is X_G
#
#  The graph-level embedding X_G captures:
#    ✓ Global structural patterns (community density, connectivity)
#    ✓ Distributional characteristics of the full graph
#    ✓ Domain knowledge encoded via H_F (text semantics)
#
#  In the paper's eq. 12:  X^G = GLM(G)
#  Our implementation:     X^G = GIN_Encoder(G, node_feats)

class GraphEncoder(nn.Module):
    def __init__(self, in_dim, hidden_dim=256, out_dim=128):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.conv1 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        ))

        self.conv2 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        ))

        self.conv3 = GINConv(nn.Sequential(
            nn.Linear(hidden_dim, out_dim),
            nn.ReLU(),
            nn.Linear(out_dim, out_dim)
        ))

        self.dropout = nn.Dropout(0.3)

    def forward(self, x, edge_index):
        h = F.relu(self.lin_in(x))

        h = self.conv1(h, edge_index)
        h = self.dropout(h)

        h = self.conv2(h, edge_index)
        h = self.dropout(h)

        h = self.conv3(h, edge_index)

        # 🔥 Graph embedding
        g = h.mean(dim=0, keepdim=True)   # (1, 128)

        # 🔥 CRITICAL FIX → node-aware X_G
        X_G = h + g                       # (N, 128)

        return h, X_G

In [ ]:
encoder = GraphEncoder(NODE_FEAT_DIM).to(DEVICE)
optimizer = torch.optim.Adam(encoder.parameters(), lr=1e-3)

def cosine(a, b):
    return F.cosine_similarity(a, b)

def contrastive_loss(h_i, h_j, h_k):
    pos = torch.exp(cosine(h_i, h_j))
    neg = torch.exp(cosine(h_i, h_k))
    return -torch.log(pos / (pos + neg + 1e-8)).mean()

from torch_geometric.utils import negative_sampling

x = node_feats.to(DEVICE)
ei = edge_index.to(DEVICE)

pos_edges = train_data.edge_label_index[
    :, train_data.edge_label == 1
].to(DEVICE)

print("Training Pipeline 3...")

for epoch in range(1, 201):
    encoder.train()
    optimizer.zero_grad()

    h, X_G = encoder(x, ei)

    # sample negatives
    neg_edges = negative_sampling(
        ei,
        num_nodes=num_nodes,
        num_neg_samples=pos_edges.shape[1]
    )

    i = pos_edges[0]
    j = pos_edges[1]
    k = neg_edges[1]

    loss = contrastive_loss(h[i], h[j], h[k])

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch} | Loss: {loss.item():.4f}")

Training Pipeline 3...
Epoch 20 | Loss: 0.4295
Epoch 40 | Loss: 0.3751
Epoch 60 | Loss: 0.3654
Epoch 80 | Loss: 0.3540
Epoch 100 | Loss: 0.3575
Epoch 120 | Loss: 0.3532
Epoch 140 | Loss: 0.3498
Epoch 160 | Loss: 0.3480
Epoch 180 | Loss: 0.3483
Epoch 200 | Loss: 0.3468


In [ ]:
#  ── Extract X_G: the graph distribution embedding ──
#
#  X_G is the graph-level embedding broadcast to all nodes.
#  In eq. 12 of the paper: X^G = GLM(G)
#  It encodes global graph distribution knowledge that guides
#  the fusion gate in Pipeline 4.
#
#  We extract two tensors:
#    node_emb_G : (N, 128)  node representations from GIN
#    X_G        : (N, 128)  graph-level embedding (same for all nodes)

encoder.eval()
with torch.no_grad():
    node_emb, X_G = encoder(x, ei)

node_emb = node_emb.cpu()
X_G      = X_G.cpu()

print("node_emb:", node_emb.shape)
print("X_G:", X_G.shape)
print("X_G same for all nodes:", torch.allclose(X_G[0], X_G[1]))


node_emb: torch.Size([2708, 128])
X_G: torch.Size([2708, 128])
X_G same for all nodes: False


In [ ]:
def evaluate(H, split_data):
    pairs = split_data.edge_label_index
    labels = split_data.edge_label.numpy()

    scores = (H[pairs[0]] * H[pairs[1]]).sum(dim=-1).numpy()

    ap  = average_precision_score(labels, scores)
    auc = roc_auc_score(labels, scores)

    return ap, auc

ap, auc = evaluate(node_emb, test_data)

print("\nPipeline 3 Results:")
print(f"AP  : {ap*100:.2f}%")
print(f"AUC : {auc*100:.2f}%")


Pipeline 3 Results:
AP  : 86.58%
AUC : 84.99%


In [ ]:
# ── Sanity checks ──────────────────────────────────

print("\n── Sanity checks ─────────────────────────────────────")
print(f"  X_G   NaN : {torch.isnan(X_G).any().item()}")
print(f"  X_G   Inf : {torch.isinf(X_G).any().item()}")
print(f"  X_G   min/max/mean : "
      f"{X_G.min():.4f} / {X_G.max():.4f} / {X_G.mean():.4f}")
print(f"  LPE   min/max/mean : "
      f"{lpe.min():.4f} / {lpe.max():.4f} / {lpe.mean():.4f}")

# Verify X_G carries graph-level (not node-level) info:
# All nodes should get the same graph embedding value
unique_rows = torch.unique(X_G, dim=0).shape[0]
print(f"\n  Unique rows in X_G : {unique_rows}  "
      f"(should be 1 for a single connected graph)")

# Quick embedding stats
print(f"\n  node_emb_G norm (mean over nodes) : "
      f"{node_emb_G.norm(dim=-1).mean():.4f}")





── Sanity checks ─────────────────────────────────────
  X_G   NaN : False
  X_G   Inf : False
  X_G   min/max/mean : -127.0276 / 165.4416 / -0.0125
  LPE   min/max/mean : -0.2991 / 0.5098 / 0.0002

  Unique rows in X_G : 2650  (should be 1 for a single connected graph)

  node_emb_G norm (mean over nodes) : 15.8132


In [ ]:
# ─ Save Pipeline 3 outputs ───────────────────────

torch.save({
    "X_G":              X_G,               # (N, 128) graph distribution emb
    "node_emb_G":       node_emb_G,        # (N, 128) GIN node embeddings
    "lpe":              lpe,               # (N,  20) Laplacian positional enc
    "graph_encoder":    graph_encoder.state_dict(),
    "link_head":        link_head.state_dict(),
    "train_data":       train_data,
    "val_data":         val_data,
    "test_data":        test_data,
    "num_nodes":        num_nodes,
    "edge_index":       edge_index,
}, "pipeline3_outputs.pt")

print("✅ X_G saved to pipeline3_outputs.pt")
print()
print("="*52)
print("  Pipeline 3 complete. Next step → Pipeline 4")
print("  (Dynamic Gate Fusion + Final Link Prediction)")
print("="*52)

# ── NOTE: Real GraphGPT replacement (for larger GPU) ─────────
#
#
# from transformers import AutoTokenizer, AutoModel
# tokenizer = AutoTokenizer.from_pretrained("HKUST-KnowComp/GraphGPT")
# model     = AutoModel.from_pretrained("HKUST-KnowComp/GraphGPT")
# model     = model.to(DEVICE)
#
# Then for each connected component g_i in the graph:
#   prompt = build_graph_prompt(g_i, node_texts)
#   X_G_i  = model.encode(prompt)
#
# The rest of Pipeline 4 stays identical since X_G dim = 128.

✅ X_G saved to pipeline3_outputs.pt

  Pipeline 3 complete. Next step → Pipeline 4
  (Dynamic Gate Fusion + Final Link Prediction)


In [ ]:
#  SFDDGNN  –  Pipeline 4: Dynamic Gate Fusion Mechanism
#  Multi-head attention fusion of H_S, H_F, X_G → Link Prediction
#  Paper: Knowledge-Based Systems 2025
#  Tested on: Google Colab  |  Dataset: Cora
# ============================================================
#
#  PREREQUISITE: All three pipeline output files must exist:
#    - pipeline1_outputs.pt  (H_S : structure embedding)
#    - pipeline2_outputs.pt  (H_F : feature embedding)
#    - pipeline3_outputs.pt  (X_G : graph distribution embedding)
#
# ============================================================


# ── CELL 1 ── Install dependencies ───────────────────────────

import subprocess, sys

def pip(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           *args, "-q"])

pip("numpy==1.26.4", "--force-reinstall")
pip("torch-geometric")
pip("scikit-learn", "scipy")

print("✅ Dependencies ready.")
print("👉 Runtime → Restart Runtime, then run Cell 2 onwards.")



✅ Dependencies ready.
👉 Runtime → Restart Runtime, then run Cell 2 onwards.


In [ ]:
import numpy as np
print(f"numpy  : {np.__version__}")

import os, random, warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.optim.lr_scheduler import StepLR

import torch_geometric
from torch_geometric.data.data import DataEdgeAttr, DataTensorAttr
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling

from sklearn.metrics import average_precision_score, roc_auc_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"torch  : {torch.__version__}")
print(f"pyg    : {torch_geometric.__version__}")
print(f"device : {DEVICE}")
print("✅ Imports done")


numpy  : 1.26.4
torch  : 2.10.0+cu128
pyg    : 2.7.0
device : cuda
✅ Imports done


In [ ]:
# Allowlist PyG classes serialised inside the .pt files
torch.serialization.add_safe_globals([
    DataEdgeAttr, DataTensorAttr, Data,
])

p1 = torch.load("pipeline1_outputs.pt",
                 map_location="cpu", weights_only=False)
p2 = torch.load("pipeline2_outputs.pt",
                 map_location="cpu", weights_only=False)
p3 = torch.load("pipeline3_outputs.pt",
                 map_location="cpu", weights_only=False)

# ── Three independent embeddings (all N×128) ──────────────────
H_S = p1["H_S"]          # structure-only  (Pipeline 1)
H_F = p2["H_F"]          # feature-only    (Pipeline 2)
X_G = p3["X_G"]          # graph knowledge (Pipeline 3)

train_data = p1["train_data"]
val_data   = p1["val_data"]
test_data  = p1["test_data"]
num_nodes  = p1["num_nodes"]
edge_index = train_data.edge_index

print(f"H_S : {H_S.shape}   (structure embedding)")
print(f"H_F : {H_F.shape}   (feature  embedding)")
print(f"X_G : {X_G.shape}   (graph knowledge)")
print(f"Nodes      : {num_nodes}")
print(f"Train edges: {edge_index.shape[1]}")
print("✅ All pipeline outputs loaded")

H_S : torch.Size([2708, 128])   (structure embedding)
H_F : torch.Size([2708, 128])   (feature  embedding)
X_G : torch.Size([2708, 128])   (graph knowledge)
Nodes      : 2708
Train edges: 8448
✅ All pipeline outputs loaded


In [ ]:
class DynamicGateFusion(nn.Module):
    """
    Dynamic gate fusion mechanism (Section 3.3, eq. 13-14).

    Args:
        d_hs    : dim of H_S  (128)
        d_hf    : dim of H_F  (128)
        d_xg    : dim of X_G  (128)
        n_heads : number of attention heads  — paper uses 3
        d_k     : dim per head               — set independently of out_dim
        out_dim : final output embedding dim (128)
        dropout : dropout rate
    """

    def __init__(self, d_hs=128, d_hf=128, d_xg=128,
                 n_heads=3, d_k=64, out_dim=128, dropout=0.3):
        super().__init__()

        self.n_heads = n_heads
        self.d_k     = d_k          # 64 per head  (paper's d_w)
        self.out_dim = out_dim

        # Concatenated dims
        d_sv = d_hs + d_hf          # [H_S || H_F] = 256
        d_q  = d_xg + d_xg          # [X_G || X_G] = 256

        # Per-head projections
        # W^Q_m : (d_q,  d_k)   Query
        # W^K_m : (d_sv, d_k)   Key
        # W^V_m : (d_sv, d_k)   Value
        self.W_Q = nn.ModuleList([
            nn.Linear(d_q,  d_k, bias=False) for _ in range(n_heads)
        ])
        self.W_K = nn.ModuleList([
            nn.Linear(d_sv, d_k, bias=False) for _ in range(n_heads)
        ])
        self.W_V = nn.ModuleList([
            nn.Linear(d_sv, d_k, bias=False) for _ in range(n_heads)
        ])

        # Output projection: n_heads * d_k → out_dim
        # e.g. 3 * 64 = 192 → 128
        self.out_proj = nn.Linear(n_heads * d_k, out_dim)

        # Add & Norm
        self.residual_proj = nn.Linear(d_sv, out_dim)
        self.layer_norm    = nn.LayerNorm(out_dim)

        self.dropout = nn.Dropout(dropout)
        self.act     = nn.Sigmoid()    # σ(·) from eq. 13

    def forward(self, H_S, H_F, X_G):
        """
        H_S : (N, 128)
        H_F : (N, 128)
        X_G : (N, 128)
        Returns H_fused : (N, out_dim=128)
        """
        SV = torch.cat([H_S, H_F], dim=-1)    # (N, 256)  keys/values source
        QQ = torch.cat([X_G, X_G], dim=-1)    # (N, 256)  query source

        head_outputs = []
        scale = self.d_k ** 0.5

        for m in range(self.n_heads):
            Q = self.act(self.W_Q[m](QQ))      # (N, d_k)
            K = self.act(self.W_K[m](SV))      # (N, d_k)
            V = self.act(self.W_V[m](SV))      # (N, d_k)

            # Attention: softmax(Q K^T / √d_k) · V
            scores   = torch.matmul(Q, K.t()) / scale   # (N, N)
            attn     = F.softmax(scores, dim=-1)         # (N, N)
            attn     = self.dropout(attn)
            head_out = torch.matmul(attn, V)             # (N, d_k)
            head_outputs.append(head_out)

        # Concat heads then project: (N, n_heads*d_k) → (N, out_dim)
        multi_head = torch.cat(head_outputs, dim=-1)     # (N, 192)
        H_proj     = self.out_proj(multi_head)           # (N, 128)

        # Add & Norm
        residual = self.residual_proj(SV)                # (N, 128)
        H_fused  = self.layer_norm(H_proj + residual)   # (N, 128)

        return H_fused


fusion_model = DynamicGateFusion(
    d_hs    = 128,
    d_hf    = 128,
    d_xg    = 128,
    n_heads = 3,     # paper: 3 heads
    d_k     = 64,    # 64 per head → 3*64=192 → projected to 128
    out_dim = 128,
    dropout = 0.3,
).to(DEVICE)

total_params = sum(p.numel() for p in fusion_model.parameters())
print(f"✅ DynamicGateFusion ready  |  params = {total_params:,}")
print(f"   n_heads={fusion_model.n_heads}  "
      f"d_k={fusion_model.d_k}  "
      f"concat_dim={fusion_model.n_heads * fusion_model.d_k}  "
      f"out_dim={fusion_model.out_dim}")

# Quick shape check
with torch.no_grad():
    dummy_hs = torch.randn(10, 128).to(DEVICE)
    dummy_hf = torch.randn(10, 128).to(DEVICE)
    dummy_xg = torch.randn(10, 128).to(DEVICE)
    out = fusion_model(dummy_hs, dummy_hf, dummy_xg)
    print(f"   Shape check: input (10,128) × 3 → output {tuple(out.shape)} ✅")


✅ DynamicGateFusion ready  |  params = 205,312
   n_heads=3  d_k=64  concat_dim=192  out_dim=128
   Shape check: input (10,128) × 3 → output (10, 128) ✅


In [ ]:
# ── Link Prediction ReadOut (eq. 14) ───────────────
#
#  p_ij = δ( g(h^G_i || h^G_j) )
#
#  where:
#    h^G_i, h^G_j = fused embeddings of the two endpoint nodes
#    g(·)         = MLP that scores the concatenated pair
#    δ(·)         = Sigmoid
#
#  Loss (eq. 14):
#  L_gate = -(1/|Γ|) Σ [ y·log(p_ij) + (1-y)·log(1-p_ij) ]

class LinkReadOut(nn.Module):
    """
    Pair-wise link prediction head.
    Input  : concatenated fused embeddings of two nodes (2*out_dim)
    Output : scalar link probability
    """
    def __init__(self, in_dim=128):
        super().__init__()
        self.g = nn.Sequential(
            nn.Linear(in_dim * 2, in_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(in_dim, in_dim // 2),
            nn.ReLU(),
            nn.Linear(in_dim // 2, 1),
        )

    def forward(self, h_i, h_j):
        """
        h_i, h_j : (B, in_dim)  fused embeddings of node pairs
        Returns  : (B,)  logits (apply sigmoid for probability)
        """
        pair_emb = torch.cat([h_i, h_j], dim=-1)   # (B, 2*in_dim)
        return self.g(pair_emb).squeeze(-1)          # (B,)


readout = LinkReadOut(in_dim=128).to(DEVICE)
print(f"✅ LinkReadOut ready  |  params = "
      f"{sum(p.numel() for p in readout.parameters()):,}")


✅ LinkReadOut ready  |  params = 41,217


In [ ]:


#  We jointly train the fusion model + readout with:
#    Optimiser : Adam, lr=1e-3, weight_decay=1e-5
#    Scheduler : StepLR, decay=0.1 every 3 epochs
#    Loss      : Binary cross-entropy (eq. 14)
#    Batch     : 1024 edge pairs
#    Early stop: patience=50 on val AP
#
#  H_S, H_F, X_G are used as FIXED inputs (pre-computed).
#  Only the fusion gate and readout weights are trained here.
#  This is consistent with the paper's decoupled training strategy.

# Move embeddings to device
H_S_dev = H_S.to(DEVICE)
H_F_dev = H_F.to(DEVICE)
X_G_dev = X_G.to(DEVICE)
ei_dev  = edge_index.to(DEVICE)

# Optimiser covers fusion gate + readout
optimizer_fuse = Adam(
    list(fusion_model.parameters()) + list(readout.parameters()),
    lr=1e-3, weight_decay=1e-5
)
scheduler_fuse = StepLR(optimizer_fuse, step_size=3, gamma=0.1)

# ── Helper: get fused embeddings then score pairs ─────────────

def score_pairs(fusion, ro, hs, hf, xg, pairs):
    """
    Run fusion gate and score a set of node pairs.
    Returns logits (B,) and fused node embeddings (N, 128).
    """
    H_fused = fusion(hs, hf, xg)                     # (N, 128)
    h_i     = H_fused[pairs[0]]                       # (B, 128)
    h_j     = H_fused[pairs[1]]                       # (B, 128)
    logits  = ro(h_i, h_j)                            # (B,)
    return logits, H_fused

# ── Helper: evaluate on a split ──────────────────────────────

def evaluate_fusion(fusion, ro, hs, hf, xg, split_data, device):
    fusion.eval(); ro.eval()
    pairs  = split_data.edge_label_index.to(device)
    labels = split_data.edge_label.float()
    with torch.no_grad():
        logits, _ = score_pairs(fusion, ro, hs, hf, xg, pairs)
        probs = torch.sigmoid(logits).cpu().numpy()
    ap  = average_precision_score(labels.numpy(), probs)
    auc = roc_auc_score(labels.numpy(), probs)
    return ap, auc

# ── Training loop ─────────────────────────────────────────────

MAX_EPOCHS = 300
PATIENCE   = 50
BATCH_SIZE = 1024

train_pairs  = train_data.edge_label_index.to(DEVICE)
train_labels = train_data.edge_label.float().to(DEVICE)

best_val_ap   = 0.0
patience_cnt  = 0
best_ckpt     = None

print(f"Training Pipeline 4  (max {MAX_EPOCHS} epochs, "
      f"patience={PATIENCE})…\n")

for epoch in range(1, MAX_EPOCHS + 1):
    fusion_model.train(); readout.train()

    # Shuffle pairs each epoch
    perm         = torch.randperm(train_pairs.shape[1], device=DEVICE)
    total_loss   = 0.0
    num_batches  = 0

    for start in range(0, train_pairs.shape[1], BATCH_SIZE):
        idx          = perm[start : start + BATCH_SIZE]
        batch_pairs  = train_pairs[:, idx]
        batch_labels = train_labels[idx]

        optimizer_fuse.zero_grad()

        logits, _ = score_pairs(
            fusion_model, readout,
            H_S_dev, H_F_dev, X_G_dev,
            batch_pairs
        )

        # L_gate: binary cross-entropy (eq. 14)
        loss = F.binary_cross_entropy_with_logits(logits, batch_labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            list(fusion_model.parameters()) + list(readout.parameters()),
            max_norm=1.0
        )
        optimizer_fuse.step()

        total_loss  += loss.item()
        num_batches += 1

    scheduler_fuse.step()
    avg_loss = total_loss / max(num_batches, 1)

    # Validate every 5 epochs
    if epoch % 5 == 0:
        val_ap, val_auc = evaluate_fusion(
            fusion_model, readout,
            H_S_dev, H_F_dev, X_G_dev,
            val_data, DEVICE
        )
        lr_now = scheduler_fuse.get_last_lr()[0]
        print(f"  Epoch {epoch:4d} | loss={avg_loss:.4f} | "
              f"val AP={val_ap*100:.2f}%  AUC={val_auc*100:.2f}% | "
              f"lr={lr_now:.2e}")

        if val_ap > best_val_ap:
            best_val_ap  = val_ap
            patience_cnt = 0
            best_ckpt = {
                "fusion":  {k: v.clone() for k, v in
                            fusion_model.state_dict().items()},
                "readout": {k: v.clone() for k, v in
                            readout.state_dict().items()},
            }
        else:
            patience_cnt += 5
            if patience_cnt >= PATIENCE:
                print(f"\n  Early stop at epoch {epoch}  "
                      f"(best val AP = {best_val_ap*100:.2f}%)")
                break

# Reload best checkpoint
if best_ckpt:
    fusion_model.load_state_dict(best_ckpt["fusion"])
    readout.load_state_dict(best_ckpt["readout"])

print("\n✅ Pipeline 4 (fusion) training complete")




Training Pipeline 4  (max 300 epochs, patience=50)…

  Epoch    5 | loss=0.2339 | val AP=78.66%  AUC=76.22% | lr=1.00e-04
  Epoch   10 | loss=0.2091 | val AP=79.27%  AUC=76.70% | lr=1.00e-06
  Epoch   15 | loss=0.2079 | val AP=79.29%  AUC=76.71% | lr=1.00e-08
  Epoch   20 | loss=0.2130 | val AP=79.29%  AUC=76.71% | lr=1.00e-09
  Epoch   25 | loss=0.2073 | val AP=79.29%  AUC=76.71% | lr=1.00e-11
  Epoch   30 | loss=0.2081 | val AP=79.29%  AUC=76.71% | lr=1.00e-13
  Epoch   35 | loss=0.2110 | val AP=79.29%  AUC=76.71% | lr=1.00e-14
  Epoch   40 | loss=0.2130 | val AP=79.29%  AUC=76.71% | lr=1.00e-16
  Epoch   45 | loss=0.2109 | val AP=79.29%  AUC=76.71% | lr=1.00e-18
  Epoch   50 | loss=0.2088 | val AP=79.29%  AUC=76.71% | lr=1.00e-19
  Epoch   55 | loss=0.2100 | val AP=79.29%  AUC=76.71% | lr=1.00e-21
  Epoch   60 | loss=0.2102 | val AP=79.29%  AUC=76.71% | lr=1.00e-23
  Epoch   65 | loss=0.2119 | val AP=79.29%  AUC=76.71% | lr=1.00e-24

  Early stop at epoch 65  (best val AP = 79.29%)


In [ ]:

test_ap, test_auc = evaluate_fusion(
    fusion_model, readout,
    H_S_dev, H_F_dev, X_G_dev,
    test_data, DEVICE
)

# Also evaluate individual pipelines for comparison (ablation)
def cosine_score(H, split_data):
    pairs  = split_data.edge_label_index
    labels = split_data.edge_label.numpy()
    s = (H[pairs[0]] * H[pairs[1]]).sum(dim=-1).numpy()
    return (average_precision_score(labels, s),
            roc_auc_score(labels, s))

s_ap,  s_auc  = cosine_score(H_S, test_data)   # structure-only
f_ap,  f_auc  = cosine_score(H_F, test_data)   # feature-only
xg_ap, xg_auc = cosine_score(X_G, test_data)   # graph-dist-only

print(f"\n{'='*58}")
print(f"  SFDDGNN – Final Results on Cora (Test Set)")
print(f"{'='*58}")
print(f"  {'Method':<28}  {'AP':>8}  {'AUC':>8}")
print(f"  {'-'*46}")
print(f"  {'Structure-only (P1)':<28}  {s_ap*100:>7.2f}%  {s_auc*100:>7.2f}%")
print(f"  {'Feature-only (P2)':<28}  {f_ap*100:>7.2f}%  {f_auc*100:>7.2f}%")
print(f"  {'Graph-dist-only (P3)':<28}  {xg_ap*100:>7.2f}%  {xg_auc*100:>7.2f}%")
print(f"  {'-'*46}")
print(f"  {'SFDDGNN (P1+P2+P3 fused)':<28}  {test_ap*100:>7.2f}%  {test_auc*100:>7.2f}%")
print(f"{'='*58}")
print(f"\n  Paper target (Table 3): AP=95.44%  AUC=94.73%")


  SFDDGNN – Final Results on Cora (Test Set)
  Method                              AP       AUC
  ----------------------------------------------
  Structure-only (P1)             60.15%    55.36%
  Feature-only (P2)               64.18%    68.76%
  Graph-dist-only (P3)            90.22%    89.43%
  ----------------------------------------------
  SFDDGNN (P1+P2+P3 fused)        80.19%    78.07%

  Paper target (Table 3): AP=95.44%  AUC=94.73%
